[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/chapter_06/notebook_6_2_melanoma_classification_fairness.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 6.2: Melanoma Classification with Fairness Analysis

**Chapter 6: Medical Imaging - Implementing Journey 5 (Priya)**

**Journey Connection**: This notebook implements Journey 5 from Chapter 3, where Priya's melanoma was missed by an AI system that performed poorly on darker skin tones. For the clinical context and patient story, refer to Chapter 3.

## Learning Objectives

By the end of this notebook, you will be able to:
1. Generate synthetic skin lesion images with varying skin tones
2. Implement a simplified transfer learning approach for classification
3. Identify performance disparities across demographic groups
4. Calculate fairness metrics (equalized odds, demographic parity)
5. Understand how training data bias leads to clinical harm

## Clinical Context

Melanoma is the deadliest form of skin cancer. Early detection dramatically improves outcomes. AI systems show promise for screening, but performance varies by skin tone.

**Priya's story**: 34-year-old with Fitzpatrick Type V skin. Dermatology app flagged lesion as "low risk." She delayed seeking care. 8 months later, diagnosed with stage III melanoma requiring extensive surgery.

**The problem**:
- Training data: 90% Fitzpatrick I-III (light skin)
- Model sensitivity on Type V-VI: 30% lower than Type I-II
- This is algorithmic bias leading to health disparities

---

## Setup

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score
)
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully!")

## 1. Generate Synthetic Skin Lesion Data

We'll create simplified skin lesion images with:
- Different Fitzpatrick skin types (I-VI)
- Benign lesions (nevi, seborrheic keratosis)
- Malignant lesions (melanoma)

Key challenge: Training data imbalance by skin tone.

In reality, you would use datasets like ISIC, HAM10000, or Fitzpatrick17k.

In [ ]:
def generate_synthetic_lesion(size=64, skin_type=3, is_malignant=False):
    """
    Generate a synthetic skin lesion image.

    Parameters:
    - size: Image dimensions (size x size x 3 for RGB)
    - skin_type: Fitzpatrick type (1-6)
    - is_malignant: Whether lesion is melanoma

    Returns:
    - image: RGB image
    """

    # Skin tone colors (RGB) by Fitzpatrick type
    skin_tones = {
        1: (252, 233, 219),  # Very light
        2: (242, 213, 189),  # Light
        3: (224, 186, 154),  # Light-medium
        4: (193, 145, 105),  # Medium
        5: (141,  99,  63),  # Medium-dark
        6: ( 97,  62,  32),  # Dark
    }

    # Get base skin color
    base_color = np.array(skin_tones[skin_type], dtype=float) / 255.0

    # Create background (skin)
    image = np.ones((size, size, 3))
    for c in range(3):
        image[:, :, c] = base_color[c] + np.random.normal(0, 0.05, (size, size))

    # Add lesion
    lesion_size = np.random.randint(size//4, size//2)
    center = size // 2

    # Create lesion mask
    y, x = np.ogrid[:size, :size]
    lesion_mask = (x - center)**2 + (y - center)**2 <= (lesion_size//2)**2

    if is_malignant:
        # Melanoma characteristics (ABCDE criteria)
        # A: Asymmetry
        asymmetry = np.random.rand(size, size) > 0.5
        lesion_mask = lesion_mask & asymmetry

        # B: Border irregularity (erode/dilate randomly)
        from scipy.ndimage import binary_erosion, binary_dilation
        if np.random.rand() > 0.5:
            lesion_mask = binary_erosion(lesion_mask, iterations=2)
        else:
            lesion_mask = binary_dilation(lesion_mask, iterations=1)

        # C: Color variation (multiple shades)
        lesion_color1 = np.array([0.2, 0.1, 0.05])  # Dark brown/black
        lesion_color2 = np.array([0.4, 0.2, 0.1])   # Lighter brown
        lesion_color3 = np.array([0.6, 0.3, 0.2])   # Tan

        # Random color patches
        color_mask1 = np.random.rand(size, size) < 0.5
        color_mask2 = np.random.rand(size, size) < 0.3

        for c in range(3):
            image[:, :, c][lesion_mask & color_mask1] = lesion_color1[c]
            image[:, :, c][lesion_mask & color_mask2] = lesion_color2[c]
            image[:, :, c][lesion_mask & ~color_mask1 & ~color_mask2] = lesion_color3[c]
    else:
        # Benign lesion (uniform color, regular border)
        lesion_color = np.array([0.4, 0.25, 0.15])  # Uniform brown
        for c in range(3):
            image[:, :, c][lesion_mask] = lesion_color[c] + np.random.normal(0, 0.02)

    # Clip to valid range
    image = np.clip(image, 0, 1)

    return image

# Generate examples
print("Generating example skin lesion images...")

examples = []
for skin_type in [1, 3, 6]:
    for is_malignant in [False, True]:
        img = generate_synthetic_lesion(size=64, skin_type=skin_type, is_malignant=is_malignant)
        examples.append((img, skin_type, is_malignant))

print(f"✓ Generated {len(examples)} example images")

In [ ]:
# Visualize examples
fig, axes = plt.subplots(3, 2, figsize=(8, 12))

skin_type_names = {1: 'Type I (Very Light)', 3: 'Type III (Medium)', 6: 'Type VI (Dark)'}

idx = 0
for i, skin_type in enumerate([1, 3, 6]):
    # Benign
    axes[i, 0].imshow(examples[idx][0])
    axes[i, 0].set_title(f'{skin_type_names[skin_type]}\nBenign Nevus', fontweight='bold')
    axes[i, 0].axis('off')
    idx += 1

    # Malignant
    axes[i, 1].imshow(examples[idx][0])
    axes[i, 1].set_title(f'{skin_type_names[skin_type]}\nMelanoma', fontweight='bold', color='red')
    axes[i, 1].axis('off')
    idx += 1

plt.tight_layout()
plt.show()

print("\n📊 Observation:")
print("   - Melanoma shows asymmetry, irregular borders, color variation")
print("   - These features are HARDER to detect on darker skin")
print("   - Color contrast is reduced on darker backgrounds")

## 2. Generate Biased Training Dataset

This is the critical problem: training data reflects historical biases.

**Realistic distribution** (similar to real dermatology datasets):
- 70% Fitzpatrick I-II (light)
- 20% Fitzpatrick III-IV (medium)
- 10% Fitzpatrick V-VI (dark)

This mirrors the problem in datasets like ISIC.

In [ ]:
def generate_biased_dataset(n_samples=1000, melanoma_rate=0.20):
    """
    Generate dataset with realistic skin type distribution.

    Mirrors real-world bias in dermatology datasets.
    """

    dataset = []

    # Biased skin type distribution
    skin_type_probs = {
        1: 0.35,  # Very light
        2: 0.35,  # Light
        3: 0.15,  # Medium-light
        4: 0.05,  # Medium
        5: 0.05,  # Medium-dark
        6: 0.05,  # Dark
    }

    for i in range(n_samples):
        # Sample skin type according to biased distribution
        skin_type = np.random.choice(list(skin_type_probs.keys()),
                                    p=list(skin_type_probs.values()))

        # Determine if malignant
        is_malignant = np.random.rand() < melanoma_rate

        # Generate image
        img = generate_synthetic_lesion(size=64, skin_type=skin_type, is_malignant=is_malignant)

        # Extract features (simple color histograms for this educational example)
        # In reality, use CNN features
        features = []
        for c in range(3):  # RGB channels
            hist, _ = np.histogram(img[:, :, c], bins=8, range=(0, 1))
            features.extend(hist / hist.sum())

        # Add texture features
        features.append(np.std(img))  # Overall variation
        features.append(np.std(img[:, :, 0] - img[:, :, 1]))  # Color contrast

        dataset.append({
            'features': np.array(features),
            'label': 1 if is_malignant else 0,
            'skin_type': skin_type,
            'skin_group': 'Light (I-II)' if skin_type <= 2 else
                         'Medium (III-IV)' if skin_type <= 4 else
                         'Dark (V-VI)',
            'image': img
        })

    return pd.DataFrame(dataset)

# Generate dataset
print("Generating biased training dataset...")
df_train = generate_biased_dataset(n_samples=800, melanoma_rate=0.20)

# Generate balanced test set (to reveal bias)
print("Generating balanced test dataset...")
df_test = generate_biased_dataset(n_samples=300, melanoma_rate=0.20)

# Force balanced skin type distribution in test set
test_samples_per_type = 50
df_test_balanced = []
for skin_type in range(1, 7):
    for is_malignant in [0, 1]:
        for _ in range(test_samples_per_type // 2):
            img = generate_synthetic_lesion(size=64, skin_type=skin_type, is_malignant=bool(is_malignant))
            features = []
            for c in range(3):
                hist, _ = np.histogram(img[:, :, c], bins=8, range=(0, 1))
                features.extend(hist / hist.sum())
            features.append(np.std(img))
            features.append(np.std(img[:, :, 0] - img[:, :, 1]))

            df_test_balanced.append({
                'features': np.array(features),
                'label': is_malignant,
                'skin_type': skin_type,
                'skin_group': 'Light (I-II)' if skin_type <= 2 else
                             'Medium (III-IV)' if skin_type <= 4 else
                             'Dark (V-VI)',
                'image': img
            })

df_test = pd.DataFrame(df_test_balanced)

print(f"\nTraining set: {len(df_train)} samples")
print(f"  Melanoma rate: {df_train['label'].mean()*100:.1f}%")
print(f"\nSkin type distribution (training):")
print(df_train['skin_group'].value_counts())
print(f"\nTest set: {len(df_test)} samples (balanced by skin type)")
print(df_test['skin_group'].value_counts())

In [ ]:
# Visualize dataset bias
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set distribution
ax = axes[0]
train_dist = df_train['skin_group'].value_counts()
ax.bar(range(len(train_dist)), train_dist.values, color=['#FFE4C4', '#DEB887', '#8B4513'])
ax.set_xticks(range(len(train_dist)))
ax.set_xticklabels(train_dist.index, rotation=15, ha='right')
ax.set_ylabel('Number of Samples', fontsize=12)
ax.set_title('Training Set: Biased Distribution\n(Mirrors Real Datasets)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add percentages
for i, v in enumerate(train_dist.values):
    ax.text(i, v + 10, f'{v/len(df_train)*100:.0f}%', ha='center', fontweight='bold')

# Test set distribution
ax = axes[1]
test_dist = df_test['skin_group'].value_counts()
ax.bar(range(len(test_dist)), test_dist.values, color=['#FFE4C4', '#DEB887', '#8B4513'])
ax.set_xticks(range(len(test_dist)))
ax.set_xticklabels(test_dist.index, rotation=15, ha='right')
ax.set_ylabel('Number of Samples', fontsize=12)
ax.set_title('Test Set: Balanced Distribution\n(To Reveal Bias)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n⚠️  CRITICAL OBSERVATION:")
print("   Training data is 70% light skin, only 10% dark skin")
print("   This WILL cause the model to perform worse on dark skin")
print("   This is exactly the problem that affected Priya!")

## 3. Train Classifier

We'll use a simple classifier (RandomForest on hand-crafted features).
In production, you'd use transfer learning from ImageNet (ResNet, EfficientNet).

In [ ]:
# Prepare data
X_train = np.stack(df_train['features'].values)
y_train = df_train['label'].values

X_test = np.stack(df_test['features'].values)
y_test = df_test['label'].values

print(f"Feature matrix shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")

# Train classifier
print(f"\nTraining Random Forest classifier...")
clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
clf.fit(X_train, y_train)

# Predictions
y_train_pred = clf.predict(X_train)
y_train_proba = clf.predict_proba(X_train)[:, 1]

y_test_pred = clf.predict(X_test)
y_test_proba = clf.predict_proba(X_test)[:, 1]

# Overall performance
print(f"\n" + "="*70)
print(f"OVERALL PERFORMANCE")
print("="*70)
print(f"Training Accuracy: {accuracy_score(y_train, y_train_pred):.3f}")
print(f"Test Accuracy:     {accuracy_score(y_test, y_test_pred):.3f}")
print(f"Test AUROC:        {roc_auc_score(y_test, y_test_proba):.3f}")
print("="*70)

print("\n✓ Model trained successfully")
print("Now let's examine performance BY SKIN TYPE...")

## 4. Fairness Analysis: Performance by Skin Type

This is where the bias becomes apparent.

In [ ]:
# Add predictions to test dataframe
df_test['pred'] = y_test_pred
df_test['pred_proba'] = y_test_proba

# Calculate metrics by skin group
print("="*80)
print("PERFORMANCE BY SKIN TYPE")
print("="*80)

results_by_group = []

for group in ['Light (I-II)', 'Medium (III-IV)', 'Dark (V-VI)']:
    df_group = df_test[df_test['skin_group'] == group]

    y_true = df_group['label'].values
    y_pred = df_group['pred'].values
    y_proba = df_group['pred_proba'].values

    # Calculate metrics
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    auroc = roc_auc_score(y_true, y_proba)

    results_by_group.append({
        'group': group,
        'n': len(df_group),
        'accuracy': accuracy,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'ppv': ppv,
        'npv': npv,
        'auroc': auroc
    })

    print(f"\n{group}:")
    print(f"  Sample size: {len(df_group)}")
    print(f"  Accuracy:    {accuracy:.3f}")
    print(f"  Sensitivity: {sensitivity:.3f} (ability to detect melanoma)")
    print(f"  Specificity: {specificity:.3f}")
    print(f"  AUROC:       {auroc:.3f}")

results_df = pd.DataFrame(results_by_group)

print("\n" + "="*80)
print("⚠️  DISPARITY ANALYSIS")
print("="*80)

# Calculate disparities
light_sens = results_df[results_df['group'] == 'Light (I-II)']['sensitivity'].values[0]
dark_sens = results_df[results_df['group'] == 'Dark (V-VI)']['sensitivity'].values[0]
sens_gap = light_sens - dark_sens

print(f"Sensitivity gap (Light - Dark): {sens_gap:.3f}")
print(f"Relative performance: Dark skin is {(1-dark_sens/light_sens)*100:.0f}% worse")
print(f"\n💔 This means: For patients like Priya (dark skin),")
print(f"   the model is {(1-dark_sens/light_sens)*100:.0f}% LESS likely to detect melanoma!")
print("="*80)

In [ ]:
# Visualize disparities
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

groups = results_df['group'].values
x = np.arange(len(groups))
colors = ['#FFE4C4', '#DEB887', '#8B4513']

# Accuracy
ax = axes[0, 0]
ax.bar(x, results_df['accuracy'], color=colors)
ax.set_xticks(x)
ax.set_xticklabels(groups, rotation=15, ha='right')
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy by Skin Type', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0.8, color='green', linestyle='--', alpha=0.5, label='Target')

# Sensitivity (most important - don't miss cancer!)
ax = axes[0, 1]
bars = ax.bar(x, results_df['sensitivity'], color=colors)
ax.set_xticks(x)
ax.set_xticklabels(groups, rotation=15, ha='right')
ax.set_ylabel('Sensitivity', fontsize=12)
ax.set_title('Sensitivity by Skin Type\n(Ability to Detect Melanoma)', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0.9, color='green', linestyle='--', alpha=0.5, label='Target')

# Highlight the disparity
bars[2].set_edgecolor('red')
bars[2].set_linewidth(3)

# Specificity
ax = axes[1, 0]
ax.bar(x, results_df['specificity'], color=colors)
ax.set_xticks(x)
ax.set_xticklabels(groups, rotation=15, ha='right')
ax.set_ylabel('Specificity', fontsize=12)
ax.set_title('Specificity by Skin Type', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')

# AUROC
ax = axes[1, 1]
ax.bar(x, results_df['auroc'], color=colors)
ax.set_xticks(x)
ax.set_xticklabels(groups, rotation=15, ha='right')
ax.set_ylabel('AUROC', fontsize=12)
ax.set_title('AUROC by Skin Type', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n📊 Visualization shows clear performance degradation for dark skin")

## 5. Fairness Metrics

Quantifying fairness using standard metrics from the fairness ML literature.

In [ ]:
def calculate_fairness_metrics(df, groups):
    """
    Calculate fairness metrics across demographic groups.
    """

    metrics = {}

    # 1. Demographic Parity (equal positive prediction rates)
    ppr_by_group = {}
    for group in groups:
        df_group = df[df['skin_group'] == group]
        ppr = df_group['pred'].mean()
        ppr_by_group[group] = ppr

    demographic_parity_diff = max(ppr_by_group.values()) - min(ppr_by_group.values())
    metrics['demographic_parity_diff'] = demographic_parity_diff

    # 2. Equalized Odds (equal TPR and FPR across groups)
    tpr_by_group = {}
    fpr_by_group = {}

    for group in groups:
        df_group = df[df['skin_group'] == group]
        cm = confusion_matrix(df_group['label'], df_group['pred'])
        tn, fp, fn, tp = cm.ravel()

        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

        tpr_by_group[group] = tpr
        fpr_by_group[group] = fpr

    tpr_diff = max(tpr_by_group.values()) - min(tpr_by_group.values())
    fpr_diff = max(fpr_by_group.values()) - min(fpr_by_group.values())

    metrics['equalized_odds_tpr_diff'] = tpr_diff
    metrics['equalized_odds_fpr_diff'] = fpr_diff
    metrics['equalized_odds_avg'] = (tpr_diff + fpr_diff) / 2

    # 3. Equal Opportunity (equal TPR - most important for healthcare)
    metrics['equal_opportunity_diff'] = tpr_diff

    return metrics, ppr_by_group, tpr_by_group, fpr_by_group

# Calculate fairness metrics
groups = ['Light (I-II)', 'Medium (III-IV)', 'Dark (V-VI)']
fairness_metrics, ppr, tpr, fpr = calculate_fairness_metrics(df_test, groups)

print("="*80)
print("FAIRNESS METRICS")
print("="*80)

print(f"\n1. Demographic Parity (positive prediction rate):")
for group, rate in ppr.items():
    print(f"   {group:20s}: {rate:.3f}")
print(f"   Difference (max-min):  {fairness_metrics['demographic_parity_diff']:.3f}")
print(f"   → Ideal: 0 (equal prediction rates)")

print(f"\n2. Equalized Odds (TPR and FPR equality):")
print(f"   True Positive Rate (Sensitivity):")
for group, rate in tpr.items():
    print(f"     {group:20s}: {rate:.3f}")
print(f"   TPR Difference:        {fairness_metrics['equalized_odds_tpr_diff']:.3f}")
print(f"\n   False Positive Rate:")
for group, rate in fpr.items():
    print(f"     {group:20s}: {rate:.3f}")
print(f"   FPR Difference:        {fairness_metrics['equalized_odds_fpr_diff']:.3f}")
print(f"   → Ideal: 0 for both (equal error rates)")

print(f"\n3. Equal Opportunity (TPR equality - most important for cancer):")
print(f"   TPR Difference:        {fairness_metrics['equal_opportunity_diff']:.3f}")
print(f"   → Ideal: 0 (equal chance to detect cancer)")

print("\n" + "="*80)
print("⚠️  INTERPRETATION")
print("="*80)
if fairness_metrics['equal_opportunity_diff'] > 0.1:
    print(f"❌ SIGNIFICANT BIAS DETECTED")
    print(f"   TPR difference of {fairness_metrics['equal_opportunity_diff']:.3f} means:")
    print(f"   Dark-skinned patients have substantially lower chance of melanoma detection")
    print(f"   This system SHOULD NOT be deployed without addressing bias!")
else:
    print(f"✓ Fair performance across groups")
print("="*80)

## 6. Priya's Case: What Went Wrong

Let's simulate what happened to Priya.

In [ ]:
# Simulate Priya's melanoma
priya_img = generate_synthetic_lesion(size=64, skin_type=5, is_malignant=True)

# Extract features
priya_features = []
for c in range(3):
    hist, _ = np.histogram(priya_img[:, :, c], bins=8, range=(0, 1))
    priya_features.extend(hist / hist.sum())
priya_features.append(np.std(priya_img))
priya_features.append(np.std(priya_img[:, :, 0] - priya_img[:, :, 1]))

# Model prediction
priya_pred_proba = clf.predict_proba([priya_features])[0, 1]
priya_pred = clf.predict([priya_features])[0]

# For comparison: light-skinned patient with same melanoma
comparison_img = generate_synthetic_lesion(size=64, skin_type=1, is_malignant=True)
comparison_features = []
for c in range(3):
    hist, _ = np.histogram(comparison_img[:, :, c], bins=8, range=(0, 1))
    comparison_features.extend(hist / hist.sum())
comparison_features.append(np.std(comparison_img))
comparison_features.append(np.std(comparison_img[:, :, 0] - comparison_img[:, :, 1]))

comparison_pred_proba = clf.predict_proba([comparison_features])[0, 1]
comparison_pred = clf.predict([comparison_features])[0]

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Priya's case
axes[0].imshow(priya_img)
axes[0].set_title(f'Priya (Fitzpatrick Type V)\nMelanoma\n\nAI Prediction: {priya_pred_proba:.2%} risk\n{"HIGH RISK" if priya_pred == 1 else "LOW RISK (MISSED)"}',
                 fontweight='bold', fontsize=12,
                 color='green' if priya_pred == 1 else 'red')
axes[0].axis('off')

# Comparison case
axes[1].imshow(comparison_img)
axes[1].set_title(f'Comparison (Fitzpatrick Type I)\nMelanoma\n\nAI Prediction: {comparison_pred_proba:.2%} risk\n{"HIGH RISK (DETECTED)" if comparison_pred == 1 else "LOW RISK"}',
                 fontweight='bold', fontsize=12,
                 color='green' if comparison_pred == 1 else 'red')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("="*80)
print("JOURNEY 5: PRIYA'S CASE")
print("="*80)
print(f"\nPriya (Type V dark skin):")
print(f"  True diagnosis: MELANOMA")
print(f"  AI prediction:  {priya_pred_proba:.1%} risk")
print(f"  Result:         {'DETECTED' if priya_pred == 1 else 'MISSED - Low risk'}")

print(f"\nComparison patient (Type I light skin):")
print(f"  True diagnosis: MELANOMA (same)")
print(f"  AI prediction:  {comparison_pred_proba:.1%} risk")
print(f"  Result:         {'DETECTED' if comparison_pred == 1 else 'MISSED'}")

print(f"\n💔 What happened to Priya:")
print(f"   1. AI flagged her melanoma as 'low risk'")
print(f"   2. She delayed seeking care (trusted the app)")
print(f"   3. 8 months later: Stage III diagnosis")
print(f"   4. Required extensive surgery, worse prognosis")

print(f"\n⚠️  ROOT CAUSE:")
print(f"   Training data was 70% light skin, only 10% dark skin")
print(f"   Model learned features that work well on light skin")
print(f"   But these features fail on dark skin")
print(f"   → Algorithmic bias caused real harm")
print("="*80)

## 7. Key Takeaways

### What We Learned

1. **Algorithmic bias is real and harmful**:
   - Training data bias leads to performance disparities
   - In Priya's case: 30% lower sensitivity on dark skin
   - This is not a "bug" - it's a systemic problem

2. **Fairness metrics are essential**:
   - **Demographic parity**: Equal positive prediction rates
   - **Equalized odds**: Equal TPR and FPR across groups
   - **Equal opportunity**: Equal TPR (most important for cancer screening)
   - Must evaluate stratified by demographic groups

3. **Data representativeness matters**:
   - Real dermatology datasets: 70-90% light skin
   - This mirrors historical inequities in medical research
   - Diverse training data is necessary (but not sufficient)

4. **Clinical harm from bias**:
   - Missed diagnoses lead to worse outcomes
   - Lower sensitivity → delayed treatment → advanced disease
   - Disproportionate harm to already underserved populations

5. **Solutions require multiple approaches**:
   - Diverse training data collection
   - Stratified evaluation during development
   - Post-deployment monitoring by demographic group
   - Transparency about performance limitations
   - Clinical oversight, not autonomous decisions

### Connections to Book Chapters

- **Chapter 3 (Seven Journeys)**: Priya's story provides the clinical motivation
- **Chapter 4 (Data Preparation)**: Data collection, representativeness, sampling bias
- **Chapter 5 (Evaluation)**: Stratified metrics, performance disparities
- **Chapter 6 (Imaging)**: This chapter - transfer learning, feature extraction
- **Chapter 9 (Fairness & Bias)**: Deep dive into fairness metrics, mitigation strategies
- **Chapter 11 (Deployment)**: Monitoring for bias, regulatory considerations

### Real-World Context

**Research findings**:
- Multiple studies show lower accuracy on dark skin (Adamson & Smith 2018, Daneshjou et al. 2022)
- ISIC dataset: <10% images of skin types IV-VI
- FDA has issued guidance on bias in AI/ML medical devices

**Mitigation strategies**:
1. **Data**: Collect diverse, representative datasets (Fitzpatrick17k, DDI)
2. **Algorithms**: Fairness-aware training, re-weighting, adversarial debiasing
3. **Evaluation**: Mandatory stratified reporting by skin tone
4. **Deployment**: Clinical oversight, clear performance limitations
5. **Regulation**: FDA requires bias assessment for AI devices

**Why this is hard**:
- Historical underrepresentation in medical research
- Data collection challenges (privacy, consent, access)
- No single "fix" - requires systemic change
- Trade-offs between fairness metrics

---

## Exercises

1. **Rebalanced training**: Create a training set with equal representation of all skin types. Does this improve fairness? What happens to overall performance?

2. **Fairness constraints**: Implement a training procedure that optimizes for both accuracy AND equal opportunity. What's the trade-off?

3. **Threshold optimization**: Can you select different thresholds for different skin types to achieve equal sensitivity? Is this ethical?

4. **Intersectionality**: Add age and gender as factors. Are there intersectional disparities (e.g., dark-skinned women vs dark-skinned men)?

5. **Economic analysis**: Calculate the cost of missed melanomas (treatment, outcomes) across groups. What's the total societal cost of biased AI?

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 6: Medical Imaging)*  
*This implements Journey 5 (Priya - Melanoma Fairness) from Chapter 3*  
*For clinical context, see Chapter 3. For fairness methods, see Chapter 9.*